# Query Bitcoin transactions in BigQuery using txid.csv

This notebook reads `data/input/txid.csv` (a single column `txid`) and queries the BigQuery public dataset `bigquery-public-data.crypto_bitcoin.transactions` for matching transactions.

Auth: ensure you've run `gcloud auth application-default login` and have access to project `bitfinex-hack-tracing` (project number 806343394594).


In [6]:
# Imports and paths
from pathlib import Path
import os
import sys
from typing import List, Optional

import pandas as pd
from dotenv import load_dotenv

BASE = Path.cwd().parent  # repo root
INPUT_DIR = BASE / 'data' / 'input'
OUTPUT_DIR = BASE / 'data' / 'output'

# Allow importing helpers from src/
sys.path.append(str((BASE / 'src').resolve()))
from bq_utils import get_bigquery_client, fetch_tx_timestamps  # helpers

BASE, INPUT_DIR, OUTPUT_DIR


(WindowsPath('c:/Users/cisco/Desktop/btc_txid_tools'),
 WindowsPath('c:/Users/cisco/Desktop/btc_txid_tools/data/input'),
 WindowsPath('c:/Users/cisco/Desktop/btc_txid_tools/data/output'))

In [7]:
# Config
load_dotenv()

# BigQuery location for the public dataset
BQ_LOCATION = os.getenv('BQ_LOCATION', 'US')

# Prefer explicit project ID; fall back to env if set
BQ_PROJECT_ID: Optional[str] = 'bitfinex-hack-tracing'
BQ_PROJECT_ID = os.getenv('BQ_PROJECT_ID', BQ_PROJECT_ID)

# Batch size for IN UNNEST(...) array parameter
BATCH_SIZE = int(os.getenv('BATCH_SIZE', '5000'))

BQ_PROJECT_ID, BQ_LOCATION, BATCH_SIZE


('bitfinex-hack-tracing', 'US', 5000)

In [8]:
# Load txids from CSV
tx_csv = INPUT_DIR / 'txid.csv'
assert tx_csv.exists(), f'Not found: {tx_csv}'
tx_df = pd.read_csv(tx_csv)
assert 'txid' in tx_df.columns, "Expected a column named 'txid'"
print('Loaded txids:', len(tx_df))
tx_df.head(10)


Loaded txids: 2072


,txid
0,002740a49e16fc127b0a58a887e5ad77cc4d5114a38aa6...
1,002eaecebf435e9bc3e2f3c4804ee6dd89c1dc342585d2...
2,0038cd4166d80a95d6bf1420beb9d16c9fb5cec19fa31f...
3,0083abb19ca6ce1ff1dd55ffa6531fe4d88d3ec884d93e...
4,00934630d88a064f60a9d294de072d5fe49620f2d6d727...
5,00d4189eca25abcf1381bff4db78db8a8edf8f11c5deb0...
6,00e43489afb85a0d3bd05f3a30b2280416d7d333d52840...
7,00fa784235855c1923e85557be5c5b5909eadcbd199352...
8,0113e56efa662e7afb75ab00d3494cbd573e6334db8b61...
9,011c0366033ac9d0a5ca5a878a3f16f65495942b2a50d1...


In [9]:
# Normalize txids to uppercase hex without 0x prefix
def normalize_txid(x) -> str:
    if pd.isna(x):
        return ''
    s = str(x).strip()
    if s.lower().startswith('0x'):
        s = s[2:]
    return s.upper()

tx_df['txid_norm'] = tx_df['txid'].apply(normalize_txid)
txids: List[str] = sorted(set(t for t in tx_df['txid_norm'].tolist() if t))
print('Unique normalized txids:', len(txids))
tx_df[['txid', 'txid_norm']].head(10)


Unique normalized txids: 2072


,txid,txid_norm
0,002740a49e16fc127b0a58a887e5ad77cc4d5114a38aa6...,002740A49E16FC127B0A58A887E5AD77CC4D5114A38AA6...
1,002eaecebf435e9bc3e2f3c4804ee6dd89c1dc342585d2...,002EAECEBF435E9BC3E2F3C4804EE6DD89C1DC342585D2...
2,0038cd4166d80a95d6bf1420beb9d16c9fb5cec19fa31f...,0038CD4166D80A95D6BF1420BEB9D16C9FB5CEC19FA31F...
3,0083abb19ca6ce1ff1dd55ffa6531fe4d88d3ec884d93e...,0083ABB19CA6CE1FF1DD55FFA6531FE4D88D3EC884D93E...
4,00934630d88a064f60a9d294de072d5fe49620f2d6d727...,00934630D88A064F60A9D294DE072D5FE49620F2D6D727...
5,00d4189eca25abcf1381bff4db78db8a8edf8f11c5deb0...,00D4189ECA25ABCF1381BFF4DB78DB8A8EDF8F11C5DEB0...
6,00e43489afb85a0d3bd05f3a30b2280416d7d333d52840...,00E43489AFB85A0D3BD05F3A30B2280416D7D333D52840...
7,00fa784235855c1923e85557be5c5b5909eadcbd199352...,00FA784235855C1923E85557BE5C5B5909EADCBD199352...
8,0113e56efa662e7afb75ab00d3494cbd573e6334db8b61...,0113E56EFA662E7AFB75AB00D3494CBD573E6334DB8B61...
9,011c0366033ac9d0a5ca5a878a3f16f65495942b2a50d1...,011C0366033AC9D0A5CA5A878A3F16F65495942B2A50D1...


In [10]:
# Initialize BigQuery client and sanity check
client = get_bigquery_client(BQ_PROJECT_ID)
_ = client.query('SELECT 1').result()
print('BigQuery client ready; project =', getattr(client, 'project', BQ_PROJECT_ID))


BigQuery client ready; project = bitfinex-hack-tracing


In [11]:
# Helper: query transactions for a list of txids in batches
from google.cloud import bigquery

SQL_TX = '''
SELECT
  TO_HEX(t.`hash`) AS txid,
  t.block_timestamp
FROM `bigquery-public-data.crypto_bitcoin.transactions` AS t
WHERE TO_HEX(t.`hash`) IN UNNEST(@txid_list)
'''

def query_transactions_for_txids(
    client: bigquery.Client,
    txids: List[str],
    *,
    batch_size: int = 5000,
    location: Optional[str] = 'US',
) -> pd.DataFrame:
    frames: list[pd.DataFrame] = []
    for i in range(0, len(txids), batch_size):
        batch = txids[i:i+batch_size]
        # Pass hex strings; convert to BYTES in SQL via FROM_HEX()
        job_config = bigquery.QueryJobConfig(
            query_parameters=[bigquery.ArrayQueryParameter('txid_list', 'STRING', batch)]
        )
        job = client.query(SQL_TX, job_config=job_config, location=location)
        df_part = job.result().to_dataframe(create_bqstorage_client=False)
        frames.append(df_part)
        print(f'Fetched {len(df_part)} rows for batch {i//batch_size+1}')
    if not frames:
        return pd.DataFrame(columns=['txid','block_timestamp'])
    out = pd.concat(frames, ignore_index=True)
    # Ensure txid is uppercase to match normalization
    out['txid'] = out['txid'].astype(str).str.upper()
    return out


In [12]:
# Run the query and show sample
# Use the proven helper that queries BigQuery with STRING txids
mapping = fetch_tx_timestamps(client, txids, batch_size=BATCH_SIZE, location=BQ_LOCATION)
tx_details = pd.DataFrame({'txid': list(mapping.keys()), 'block_timestamp': list(mapping.values())})
tx_details['txid'] = tx_details['txid'].astype(str).str.upper()
print('Total matched tx rows:', len(tx_details))
tx_details.head(10)


BadRequest: 400 No matching signature for function TO_HEX
  Argument types: STRING
  Signature: TO_HEX(BYTES)
    Argument 1: Unable to coerce type STRING to expected type BYTES at [4:15]; reason: invalidQuery, location: query, message: No matching signature for function TO_HEX
  Argument types: STRING
  Signature: TO_HEX(BYTES)
    Argument 1: Unable to coerce type STRING to expected type BYTES at [4:15]

Location: US
Job ID: 5f14f93b-b444-4ffb-b70b-27ad2007054c


In [ ]:
# Save results
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
out_path = OUTPUT_DIR / 'txid_transactions.csv'
tx_details.to_csv(out_path, index=False)
out_path


NameError: name 'tx_details' is not defined